# Debug notebook for `collect_iterres.py`

This notebook lets you call the script's functions interactively without running it as a CLI script.

**Why a mock `args` is needed:**  
Several functions (`main`, `plot_graphs`) read the module-level `args` variable that is normally
populated by `argparse`. In a notebook `parse_arguments()` would try to parse Jupyter's own
kernel arguments and crash. The cell below injects a fake one instead.

In [1]:
import sys, argparse, os
from paths import scripts_path, data_prod_path, path_to_nn_runs
import numpy as np
import pandas as pd

# ── Point at the directory that contains collect_iterres.py,
#    paths.py, and analysis.py. Adjust as needed. ──────────────────────────
SCRIPT_DIR = scripts_path
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)

# ── Build a mock args namespace that mirrors every argparse argument.
#    Tweak the values here to control behaviour, exactly as you would
#    pass flags on the command line. ──────────────────────────────────────
mock_args = argparse.Namespace(
    base_dir        = None,          # will use path_to_nn_runs from paths.py
    out_dir         = "/tmp/debug_out/",
    hk_lookup_path  = None,
    weight_pfi      = False,  # True  → sort by WPFI (PFI × test_accuracy)
                              # False → sort by PFI (or UPS if PFI absent)
    top_kmers       = 500,
    filter_harsh    = False,
    show_cm_bar_percentage = False,
    x_col           = None,
    hue_col         = None,
    group_x_col     = None,
    group_hue_col   = None,
    highlight_multi = False,
    network_top_kmers = 50,
)

nn_run_dir_name = "IterExcl_encoded_sketches_n500_k12"
nn_dir = path_to_nn_runs + nn_run_dir_name + "/"
print("NN directory exists:", os.path.isdir(nn_dir))
if not os.path.isdir(nn_dir):
    raise FileNotFoundError(f"NN directory not found: {nn_dir}")
partition_folder = nn_dir + "cluster_b0_p0_run1/"
print("Partition folder exists:", os.path.isdir(partition_folder))

# ── Import the module and immediately overwrite its global `args`.
#    This must happen before calling any function that references `args`. ──
import collect_iterres as cr
cr.args = mock_args

print("Module loaded. cr.args is now:", cr.args)

NN directory exists: True
Partition folder exists: True


/net/domus/home/people/s215045/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Module loaded. cr.args is now: Namespace(base_dir=None, out_dir='/tmp/debug_out/', hk_lookup_path=None, weight_pfi=False, top_kmers=500, filter_harsh=False, show_cm_bar_percentage=False, x_col=None, hue_col=None, group_x_col=None, group_hue_col=None, highlight_multi=False, network_top_kmers=50)


## Standalone utilities — no `args` dependency

In [ ]:
# ── correct_deci_number ───────────────────────────────────────────────────
# Converts values like 568 → 0.568
print(cr.correct_deci_number(568))   # expect 0.568
print(cr.correct_deci_number(0.75))  # expect 0.75

In [ ]:
# ── calculate_unified_score ───────────────────────────────────────────────
sample_metrics = {
    'test_balanced_accuracy': 0.82,
    'f1': 0.79,
    'unseen_test_balanced_accuracy': 0.71,
}
print("UPS:", cr.calculate_unified_score(sample_metrics))  # expect ~0.763

## Parse a single log file

In [ ]:
# ── extract_metrics_from_log ──────────────────────────────────────────────
LOG_PATH = "/path/to/your/log_run_something.txt"  # <-- change this

result = cr.extract_metrics_from_log(LOG_PATH)
if result:
    metrics, run_info = result
    print("Metrics:", metrics)
    print("Run info:", run_info)
else:
    print("Parsing failed — check the log format.")

## MetricPlottingUtils — plot from a hand-built DataFrame

Useful for testing the plotting code against a small synthetic dataset
without having to scan an entire run directory.

In [ ]:
OUT = "/tmp/debug_out/"
os.makedirs(OUT, exist_ok=True)

# Minimal synthetic run table — add / remove columns to test edge cases
df_test = pd.DataFrame({
    'folder':                    ['run_a', 'run_b', 'run_c'],
    'n':                         [500,     500,     1000   ],
    'k':                         [12,      20,      12     ],
    'test_accuracy':             [0.81,    0.76,    0.84   ],
    'test_balanced_accuracy':    [0.79,    0.73,    0.82   ],
    'unseen_test_accuracy':      [0.70,    0.65,    0.74   ],
    'unseen_test_balanced_accuracy': [0.68, 0.63,   0.72  ],
    'precision':                 [0.80,    0.74,    0.83   ],
    'recall':                    [0.78,    0.72,    0.81   ],
    'f1':                        [0.79,    0.73,    0.82   ],
    'TN':                        [90,      85,      95     ],
    'FN':                        [10,      15,       8     ],
    'FP':                        [ 8,      12,       6     ],
    'TP':                        [92,      88,      97     ],
    'train_pairs':               [1000,    1000,    1200   ],
    'test_pairs':                [200,     200,     240    ],
    'test_train_ratio':          [0.2,     0.2,     0.2    ],
    'status':                    [True,    True,    True   ],
})

plotter = cr.MetricPlottingUtils(df=df_test, outdir=OUT)

# Individual plot helpers (no args dependency)
plotter._plot_bars(
    x_col='n', y_col='test_balanced_accuracy', hue_col='k',
    title='Balanced accuracy by n/k (test)',
    ylabel='Balanced Accuracy',
    outpath=OUT + 'test_balanced_acc.png'
)
print("Plot saved to", OUT + 'test_balanced_acc.png')

In [ ]:
# ── Call plot_graphs() — requires args.show_cm_bar_percentage ─────────────
# mock_args already sets it to False, so this is safe to run as-is.
plotter.plot_graphs()

## GAPlottingUtils — test annotation plots

`GAPlottingUtils` now takes `sort_by` at construction time (`'PFI'`, `'WPFI'`, or `'UPS'`).
This controls both the kmer ordering in the network plot and the y-axis in
`plot_kmer_against_ups_or_pfi`.

In [ ]:
# Build a tiny synthetic annotation dataframe (mirrors what GA.batch_bact_annotate returns)
annot_df = pd.DataFrame({
    'decoded_kmer': ['ATGCGT', 'CCGATG', 'TTACGG', 'ATGCGT', 'GGCAAT'],
    'bact':         ['E.coli', 'E.coli', 'Salmonella', 'Salmonella', 'E.coli'],
    'gene':         ['geneA', 'geneB', 'geneA', 'geneC', 'geneB'],
    'kmer_in_seq':  [5, 3, 7, 2, 4],
    'PFI':          [0.81, 0.74, 0.68, 0.90, 0.55],
    'UPS':          [0.78, 0.70, 0.65, 0.88, 0.52],
})

# sort_by='PFI' matches the default in the updated script
ga_plotter = cr.GAPlottingUtils(df=annot_df, outdir=OUT, sort_by='PFI')

ga_plotter.plot_top_genes(annot_df, entity_type='bacterium', title_suffix='(test)')
ga_plotter.plot_kmer_distribution(annot_df, entity_type='bacterium', title_suffix='(test)')
# plot_kmer_against_ups_or_pfi no longer takes sort_by — it uses self.sort_by
ga_plotter.plot_kmer_against_ups_or_pfi(annot_df, entity_type='bacterium')
print("Gene plots saved.")

## Run main() end-to-end

Reads from a real run directory. Adjust `base_dir` and `out_dir` below.

In [2]:
# Update mock_args if you want different settings for this run
cr.args.top_kmers   = 200
cr.args.weight_pfi  = True
cr.args.filter_harsh = False

cr.main(
    base_dir = nn_dir,
    outdir   = data_prod_path + "TEST/" + nn_run_dir_name+ "/",
    x_col    = None,
    hue_col  = None,
    group_x_col  = None,
    group_hue_col = None,
)

Scanning directory: /net/node07/home/projects/s215045/PredictPhagePPI/nn_runs/IterExcl_encoded_sketches_n500_k12/
Created output directory: /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/TEST/IterExcl_encoded_sketches_n500_k12/
[2026-05-23 14:57:18]  collect_iterres started. Scanning /net/node07/home/projects/s215045/PredictPhagePPI/nn_runs/IterExcl_encoded_sketches_n500_k12/ for log files.


Processing folders:   0%|          | 0/36 [00:00<?, ?it/s]


[2026-05-23 14:57:18] Processing folder: cluster_b0_p0_run1
Loaded hk_lookup JSON file from /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches/hk_lookup_n500_k12.json
Deduced hk_lookup from log info for cluster_b0_p0_run1 using path: /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches/hk_lookup_n500_k12.json
Found expected interactions file: top_interaction_pairs_expected_interactions.csv in folder: cluster_b0_p0_run1
Skipping PFI calculation for cluster_b0_p0_run1. Reason: top_kmers=True, hk_lookup=True

[2026-05-23 14:57:18] Folder processed: cluster_b0_p0_run1
##################################################


Processing folders:   3%|▎         | 1/36 [00:00<00:05,  6.91it/s]


[2026-05-23 14:57:18] Processing folder: cluster_b0_p1_run1
Loaded hk_lookup JSON file from /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches/hk_lookup_n500_k12.json
Deduced hk_lookup from log info for cluster_b0_p1_run1 using path: /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches/hk_lookup_n500_k12.json


...

Processing folders: 100%|██████████| 36/36 [00:07<00:00,  4.61it/s]

Skipping PFI calculation for cluster_b5_p5_run1. Reason: top_kmers=True, hk_lookup=True

[2026-05-23 14:57:26] Folder processed: cluster_b5_p5_run1
##################################################
Sorted top_kmers_df by weighted PFI score (expected interaction score scaled by test accuracy).
[2026-05-23 14:57:26] Extracted metrics from 36 log files.
Unable to process group_x_col: None, error: 'NoneType' object has no attribute 'lower'. Defaulting to no grouping.
All runs have the same 'n' & 'k' value. Results will be plotted based on rows and not grouped by 'n' and 'k'.
Using x_col: folder, hue_col: None
Dataframe length: 36. Recognized metrics:
  file_path: str
  test_accuracy: float64
  test_balanced_accuracy: float64
  precision: float64
  recall: float64
  f1: float64
  TN: int64
  FN: int64
  FP: int64
  TP: int64
  train_pairs: int64
  test_pairs: int64
  test_train_ratio: float64
  n: int64
  k: int64
  unseen_test_accuracy: float64
  unseen_test_balanced_accuracy: float64
  s

                                            file_path  test_accuracy  \
0   /net/node07/home/projects/s215045/PredictPhage...         0.8787   
1   /net/node07/home/projects/s215045/PredictPhage...         0.8907   
2   /net/node07/home/projects/s215045/PredictPhage...         0.8698   
3   /net/node07/home/projects/s215045/PredictPhage...         0.8907   
4   /net/node07/home/projects/s215045/PredictPhage...         0.9251   
5   /net/node07/home/projects/s215045/PredictPhage...         0.8737   
6   /net/node07/home/projects/s215045/PredictPhage...         0.8310   
7   /net/node07/home/projects/s215045/PredictPhage...         0.9075   
8   /net/node07/home/projects/s215045/PredictPhage...         0.8932   
9   /net/node07/home/projects/s215045/PredictPhage...         0.9030   
10  /net/node07/home/projects/s215045/PredictPhage...         0.8867   
11  /net/node07/home/projects/s215045/PredictPhage...         0.8375   
12  /net/node07/home/projects/s215045/PredictPhage...         0.

Annotating phage-kmer pairs: 100%|██████████| 200/200 [00:26<00:00,  7.64it/s]


[2026-05-23 14:58:26] Script execution completed.


---
## Per-partition GAPlottingUtils

The cells below run the full annotation + GAPlottingUtils pipeline on **each partition folder
individually**, rather than pooling everything together as `main()` does.

Each partition gets its own subdirectory under `BASE_OUT` so the plots never overwrite each other.

**Pipeline per folder:**
1. Parse the log file → extract metrics + run_info
2. Calculate UPS from those metrics; attach `UPS` and `test_accuracy` to `df_kmers`
3. Load `pair_kmers.csv`; rename `expected_interaction_score` → `PFI` if present
4. Optionally compute `WPFI = PFI × test_accuracy` (mirrors `weight_pfi` flag)
5. Resolve `sort_by`: `WPFI` → `PFI` → `UPS` (same priority as `main()`)
6. Split into `bact_df` / `phage_df` using the `bact_*` / `phage_*` column prefixes
   and extract `hash` from the `pair` column — matching the updated split logic in the script
7. Subset to top-k kmers via `balanced_top_k`
8. Annotate with `GeneAnalysis`
9. Instantiate `GAPlottingUtils(sort_by=sort_by)` and run all plot methods

In [7]:
def resolve_sort_by(df_kmers: pd.DataFrame, weight_pfi: bool) -> tuple:
    """
    Determine the score column and title suffix to use for sorting,
    following the same priority logic as main():
      weight_pfi=True  → WPFI (PFI × test_accuracy) if possible, else PFI
      weight_pfi=False → PFI if present, else UPS

    Returns (sort_by: str, title_suffix: str, df_kmers: pd.DataFrame)
    with WPFI column added to df_kmers when applicable.
    """
    # Normalise column name produced by older runs
    if 'expected_interaction_score' in df_kmers.columns and 'PFI' not in df_kmers.columns:
        df_kmers = df_kmers.rename(columns={'expected_interaction_score': 'PFI'})

    if 'PFI' in df_kmers.columns:
        if weight_pfi:
            if 'test_accuracy' in df_kmers.columns:
                df_kmers['WPFI'] = df_kmers['PFI'] * df_kmers['test_accuracy']
                return 'WPFI', '(WPFI)', df_kmers
            else:
                print("Warning: 'test_accuracy' missing — cannot weight PFI. Falling back to PFI.")
        return 'PFI', '(PFI)', df_kmers

    return 'UPS', '(UPS)', df_kmers


def split_bact_phage(df_kmers: pd.DataFrame) -> tuple:
    """
    Split a pair_kmers DataFrame into bact_df and phage_df using the
    bact_* / phage_* column prefixes and hash extraction from `pair`,
    exactly as main() does in the updated script.

    Score columns (PFI, UPS, test_accuracy, WPFI) are carried through
    whichever are present.

    Returns (bact_df, phage_df).
    """
    score_cols = [c for c in ('PFI', 'UPS', 'test_accuracy', 'WPFI') if c in df_kmers.columns]
    base_cols  = score_cols + ['folder']

    # ── Bacterium ─────────────────────────────────────────────────────────
    bact_df = df_kmers[['bact_entity', 'bact_organism', 'bact_decoded_kmer'] + base_cols].copy()
    bact_df = bact_df.rename(columns={
        'bact_entity':      'entity',
        'bact_organism':    'organism',
        'bact_decoded_kmer':'decoded_kmer',
    })
    if 'pair' in df_kmers.columns:
        try:
            bact_df.insert(0, 'hash',
                df_kmers['pair'].str.extract(r'np\.int64\((\d+)\)')[0].astype('int64'))
        except Exception as e:
            print(f"Could not extract bact hash from 'pair' column: {e}")

    # ── Phage ─────────────────────────────────────────────────────────────
    phage_df = df_kmers[['phage_entity', 'phage_organism', 'phage_decoded_kmer'] + base_cols].copy()
    phage_df = phage_df.rename(columns={
        'phage_entity':      'entity',
        'phage_organism':    'organism',
        'phage_decoded_kmer':'decoded_kmer',
    })
    if 'pair' in df_kmers.columns:
        try:
            phage_df.insert(0, 'hash',
                df_kmers['pair'].str.extract(r'np\.int64\(\d+\).*?np\.int64\((\d+)\)')[0].astype('int64'))
        except Exception as e:
            print(f"Could not extract phage hash from 'pair' column: {e}")

    return bact_df, phage_df


def plot_partition(folder_path, folder_name, base_out,
                   top_kmers=500, network_top_kmers=50,
                   hk_lookup_path=None, weight_pfi=False):
    """
    Run GAPlottingUtils plots for a single partition folder.

    Parameters
    ----------
    folder_path      : str  – absolute path to the partition directory
    folder_name      : str  – name used for plot titles and the output subdirectory
    base_out         : str  – root output dir; a per-partition subfolder is created inside it
    top_kmers        : int  – passed to balanced_top_k
    network_top_kmers: int  – passed to plot_kmer_gene_network
    hk_lookup_path   : str  – explicit path to a hk_lookup JSON; None → deduced from run_info
    weight_pfi       : bool – mirrors the --weight_pfi CLI flag:
                              True  → sort by WPFI (PFI × test_accuracy)
                              False → sort by PFI if available, else UPS

    Returns
    -------
    dict with keys 'bact', 'phage' (annotated DataFrames), 'metrics', 'ups', 'sort_by'
    or None if the folder could not be processed.
    """
    outdir = os.path.join(base_out, folder_name, "")
    os.makedirs(outdir, exist_ok=True)

    # ── 1. Parse log file ─────────────────────────────────────────────────
    metrics, run_info = None, None
    for fname in os.listdir(folder_path):
        if fname.endswith(".txt") and "log_run" in fname.lower():
            result = cr.extract_metrics_from_log(os.path.join(folder_path, fname))
            if result:
                metrics, run_info = result
                break

    if metrics is None:
        print(f"[{folder_name}] No parseable log file — skipping.")
        return None

    # ── 2. UPS and test_accuracy (attached per-row, same as main()) ───────
    ups           = cr.calculate_unified_score(metrics)
    test_accuracy = metrics.get('test_accuracy', None)

    # ── 3. Load expected_interactions.csv ─────────────────────────────────
    kmers_path = None
    for fname in os.listdir(folder_path):
        if fname.endswith("expected_interactions.csv"):
            kmers_path = os.path.join(folder_path, fname)
            break

    if kmers_path is None:
        print(f"[{folder_name}] No expected_interactions.csv — skipping.")
        return None

    df_kmers = pd.read_csv(kmers_path)
    df_kmers['folder']        = folder_name
    df_kmers['UPS']           = ups
    df_kmers['test_accuracy'] = test_accuracy

    # ── 4. Resolve hk_lookup ─────────────────────────────────────────────
    if hk_lookup_path:
        cr.open_hk_lookup(hk_lookup_path, reverse=True)   # side-effect: cached in cr
    elif run_info:
        try:
            dir_name = "encoded_sketches" if run_info.get('use_encoded') else "SM_sketches"
            if run_info.get('data2'):
                dir_name += "_data2"
            hk_path = os.path.join(
                cr.data_prod_path, dir_name,
                f"hk_lookup_n{metrics['n']}_k{metrics['k']}.json"
            )
            cr.open_hk_lookup(hk_path, reverse=True)
        except Exception as e:
            print(f"[{folder_name}] Could not resolve hk_lookup: {e}")

    # ── 5. Determine score column (WPFI / PFI / UPS) ─────────────────────
    sort_by, title_suffix, df_kmers = resolve_sort_by(df_kmers, weight_pfi=weight_pfi)
    print(f"[{folder_name}] Sorting by '{sort_by}'")

    # ── 6. Split into bact_df / phage_df (updated column-prefix logic) ───
    required_bact_cols  = {'bact_entity', 'bact_organism', 'bact_decoded_kmer'}
    required_phage_cols = {'phage_entity', 'phage_organism', 'phage_decoded_kmer'}

    if not required_bact_cols.issubset(df_kmers.columns) and \
       not required_phage_cols.issubset(df_kmers.columns):
        print(f"[{folder_name}] pair_kmers.csv is missing bact_*/phage_* columns — skipping.")
        return None

    bact_df, phage_df = split_bact_phage(df_kmers)

    # ── 7. Subset to top_kmers per entity ────────────────────────────────
    if not bact_df.empty and sort_by in bact_df.columns:
        bact_df = cr.balanced_top_k(
            bact_df, group_cols=['entity'], sort_col=sort_by, total_k=top_kmers
        )
    if not phage_df.empty and sort_by in phage_df.columns:
        phage_df = cr.balanced_top_k(
            phage_df, group_cols=['entity'], sort_col=sort_by, total_k=top_kmers
        )

    # ── 8. Annotate ───────────────────────────────────────────────────────
    GA = cr.GeneAnalysis()
    bact_annot  = pd.DataFrame()
    phage_annot = pd.DataFrame()

    if not bact_df.empty:
        try:
            # bact_annot = GA.batch_bact_annotate(
            #     bkmers     = bact_df['decoded_kmer'].tolist(),
            #     bact_names = bact_df['entity'].tolist(),
            #     data_prod_path = cr.data_prod_path,
            # )
            bact_annot = GA.batch_bact_annotate(
                bact_df=bact_df, 
                kmer_col='decoded_kmer', 
                entity_col='entity', 
                data_prod_path=data_prod_path
            )
            print(f"[{folder_name}] Bacterium annotation: {len(bact_annot)} rows")
        except Exception as e:
            print(f"[{folder_name}] Bacterium annotation failed: {e}")

    if not phage_df.empty:
        try:
            phage_annot = GA.batch_phage_annotate(
                phage_df=phage_df, 
                kmer_col='decoded_kmer', 
                entity_col='entity', 
                data_prod_path=data_prod_path
            )
            print(f"[{folder_name}] Phage annotation: {len(phage_annot)} rows")
        except Exception as e:
            print(f"[{folder_name}] Phage annotation failed: {e}")

    # ── 9. Plot ───────────────────────────────────────────────────────────
    # GAPlottingUtils now receives sort_by at construction time
    ga_plotter = cr.GAPlottingUtils(df=df_kmers, outdir=outdir, sort_by=sort_by)
    bact_df_agg = bact_annot.groupby("gene")["kmer_in_seq"].agg(lambda x: ';'.join(sorted(set([str(v) for v in x if pd.notna(v)])))).reset_index()
    display(bact_df_agg)
    display(bact_annot)

    if not bact_annot.empty:
        ga_plotter.plot_top_genes(
            bact_annot, entity_type='bacterium', title_suffix=f'{title_suffix} ({folder_name})')
        ga_plotter.plot_kmer_distribution(
            bact_annot, entity_type='bacterium', title_suffix=f'{title_suffix} ({folder_name})')
        ga_plotter.plot_kmer_gene_network(
            bact_annot, entity_type='bacterium', top_kmers=network_top_kmers)
        # plot_kmer_against_ups_or_pfi uses self.sort_by — no argument needed
        ga_plotter.plot_kmer_against_ups_or_pfi(bact_annot, entity_type='bacterium')

    if not phage_annot.empty:
        ga_plotter.plot_top_genes(
            phage_annot, entity_type='phage', title_suffix=f'{title_suffix} ({folder_name})')
        ga_plotter.plot_kmer_distribution(
            phage_annot, entity_type='phage', title_suffix=f'{title_suffix} ({folder_name})')
        ga_plotter.plot_kmer_gene_network(
            phage_annot, entity_type='phage', top_kmers=network_top_kmers)
        ga_plotter.plot_kmer_against_ups_or_pfi(phage_annot, entity_type='phage')

    if bact_annot.empty and phage_annot.empty:
        print(f"[{folder_name}] Both annotation DataFrames empty — no plots generated.")
    else:
        print(f"[{folder_name}] Plots saved to: {outdir}")

    return {
        'bact':    bact_annot,
        'phage':   phage_annot,
        'metrics': metrics,
        'ups':     ups,
        'sort_by': sort_by,
    }

In [ ]:
# ── Configure and run the per-partition loop ──────────────────────────────
BASE_DIR = nn_dir
BASE_OUT = data_prod_path + "TEST/" + nn_run_dir_name + "/"

# Optional: restrict to a subset of folders for a quick test
# ONLY_FOLDERS = {'cluster_1_PhageA', 'cluster_2_PhageB'}  # set to None to process all
ONLY_FOLDERS = {"cluster_b0_p0_run1"}

TOP_KMERS         = 200
NETWORK_TOP_KMERS = 50
HK_LOOKUP_PATH    = None   # or a path string to override auto-deduction
WEIGHT_PFI        = False  # set True to sort by WPFI instead of PFI

partition_results = {}

for folder_name in sorted(os.listdir(BASE_DIR)):
    if ONLY_FOLDERS and folder_name not in ONLY_FOLDERS:
        continue
    folder_path = os.path.join(BASE_DIR, folder_name)
    if not os.path.isdir(folder_path):
        continue

    partition_results[folder_name] = plot_partition(
        folder_path       = folder_path,
        folder_name       = folder_name,
        base_out          = BASE_OUT,
        top_kmers         = TOP_KMERS,
        network_top_kmers = NETWORK_TOP_KMERS,
        hk_lookup_path    = HK_LOOKUP_PATH,
        weight_pfi        = WEIGHT_PFI,
    )

succeeded = [k for k, v in partition_results.items() if v is not None]
print(f"\nDone. {len(succeeded)}/{len(partition_results)} partitions processed successfully.")

Loaded hk_lookup JSON file from /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/encoded_sketches/hk_lookup_n500_k12.json
[cluster_b0_p0_run1] Sorting by 'PFI'


Annotating bacteria-kmer pairs: 100%|██████████| 50/50 [00:08<00:00,  5.78it/s]


[cluster_b0_p0_run1] Bacterium annotation: 16 rows


Annotating phage-kmer pairs: 100%|██████████| 50/50 [00:06<00:00,  7.28it/s]

[cluster_b0_p0_run1] Phage annotation: 0 rows


,gene,kmer_in_seq
0,patZ,CGCTGCAGAATG


,bact,locus_tag,kmer_in_seq,length_bp,gene,product,UPS,PFI
0,J99_22,KCDMCKIA_03467,CGCTGCAGAATG,2652,patZ,NaN,0.4265,0.000358
1,J99_22,KCDMCKIA_03467,CGCTGCAGAATG,2652,patZ,Peptidyl-lysine N-acetyltransferase PatZ,0.4265,0.000358
2,J99_22,KCDMCKIA_03467,CGCTGCAGAATG,2652,patZ,NaN,0.4265,0.000358
3,J99_22,KCDMCKIA_03467,CGCTGCAGAATG,2652,patZ,Peptidyl-lysine N-acetyltransferase PatZ,0.4265,0.000358
4,J99_22,KCDMCKIA_03467,CGCTGCAGAATG,2652,patZ,NaN,0.4265,0.000358
5,J99_22,KCDMCKIA_03467,CGCTGCAGAATG,2652,patZ,Peptidyl-lysine N-acetyltransferase PatZ,0.4265,0.000358
6,J99_22,KCDMCKIA_03467,CGCTGCAGAATG,2652,patZ,NaN,0.4265,0.000358
7,J99_22,KCDMCKIA_03467,CGCTGCAGAATG,2652,patZ,Peptidyl-lysine N-acetyltransferase PatZ,0.4265,0.000358
8,J99_22,KCDMCKIA_03467,CGCTGCAGAATG,2652,patZ,NaN,0.4265,0.000358
9,J99_22,KCDMCKIA_03467,CGCTGCAGAATG,2652,patZ,Peptidyl-lysine N-acetyltransferase PatZ,0.4265,0.000358


Ordering kmers by PFI (descending, aggregated by mean).
[cluster_b0_p0_run1] Plots saved to: /net/node07/home/projects/s215045/PredictPhagePPI/data_prod/TEST/IterExcl_encoded_sketches_n500_k12/cluster_b0_p0_run1/

Done. 1/1 partitions processed successfully.


In [8]:
# ── Inspect results for a single partition without re-running ─────────────
# Change the name below to any key from partition_results.
INSPECT = succeeded[0] if succeeded else None

if INSPECT:
    r = partition_results[INSPECT]
    print(f"=== {INSPECT} ===")
    print(f"UPS:     {r['ups']}")
    print(f"sort_by: {r['sort_by']}")
    print(f"Metrics: {r['metrics']}\n")

    if not r['bact'].empty:
        print("Bacterium annotation (head):")
        display(r['bact'].head())

    if not r['phage'].empty:
        print("Phage annotation (head):")
        display(r['phage'].head())
else:
    print("No successful partitions to inspect.")

=== cluster_b0_p0_run1 ===
UPS:     0.4265
sort_by: PFI
Metrics: {'file_path': '/net/node07/home/projects/s215045/PredictPhagePPI/nn_runs/IterExcl_encoded_sketches_n500_k12/cluster_b0_p0_run1/log_run1.txt', 'test_accuracy': 0.8787, 'test_balanced_accuracy': 0.58, 'precision': 0.0694, 'recall': 0.2632, 'f1': 0.1099, 'TN': 582, 'FN': 14, 'FP': 67, 'TP': 5, 'train_pairs': 1862, 'test_pairs': 668, 'test_train_ratio': 0.3588, 'n': 500, 'k': 12, 'unseen_test_accuracy': 0.9792, 'unseen_test_balanced_accuracy': 0.5, 'status': True}

Bacterium annotation (head):


,bact,locus_tag,kmer_in_seq,length_bp,gene,product,UPS,PFI
0,J99_22,KCDMCKIA_03467,CGCTGCAGAATG,2652,patZ,NaN,0.4265,0.000358
1,J99_22,KCDMCKIA_03467,CGCTGCAGAATG,2652,patZ,Peptidyl-lysine N-acetyltransferase PatZ,0.4265,0.000358
2,J99_22,KCDMCKIA_03467,CGCTGCAGAATG,2652,patZ,NaN,0.4265,0.000358
3,J99_22,KCDMCKIA_03467,CGCTGCAGAATG,2652,patZ,Peptidyl-lysine N-acetyltransferase PatZ,0.4265,0.000358
4,J99_22,KCDMCKIA_03467,CGCTGCAGAATG,2652,patZ,NaN,0.4265,0.000358


In [9]:
# ── Quick score summary across all partitions ─────────────────────────────
summary = pd.DataFrame([
    {
        'partition': k,
        'sort_by':   v['sort_by'],
        'ups':       v['ups'],
        **{m: v['metrics'].get(m) for m in
           ['test_balanced_accuracy', 'unseen_test_balanced_accuracy', 'f1']}
    }
    for k, v in partition_results.items() if v is not None
]).sort_values('ups', ascending=False)

display(summary)

,partition,sort_by,ups,test_balanced_accuracy,unseen_test_balanced_accuracy,f1
0,cluster_b0_p0_run1,PFI,0.4265,0.58,0.5,0.1099
